In [15]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torch.optim import AdamW
import random

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
generator= torch.Generator().manual_seed(SEED)
rng = np.random.default_rng(SEED)


transform = transforms.Compose([
    transforms.ToTensor(),                                        # Pretvara sliku u PyTorch tenzor (vrednosti 0.0 - 1.0)
    transforms.Lambda(lambda img: torch.rot90(img, 1, [1, 2])),   # Rotacija da slova ne budu "na bok"
    transforms.Lambda(lambda img: torch.flip(img, [2])),          # Horizontalno ogledalo ako je potrebno
    transforms.Normalize((0.5,), (0.5,))                          # Normalizacija na opseg [-1, 1]
])

train_dataset = datasets.EMNIST(
    root='./data', 
    split='byclass', 
    train=True, 
    download=True, 
    transform=transform
)
test_dataset = datasets.EMNIST(
    root='./data',
    split='byclass',
    train= False,
    download=True,
    transform=transform   
)

BATCH_SIZE=256


n_test_total= len(test_dataset)
n_calib = 2000
n_id_eval = 2000

assert n_calib + n_id_eval <= n_test_total, "Nema dovoljno uzoraka u test_dataset-u"

perm=torch.randperm(n_test_total, generator=generator).tolist()
calib_ind = perm[:n_calib]
id_eval_ind =perm[n_calib:n_calib+n_id_eval]
rest_test_ind=perm[n_id_eval+n_calib :]

calib_dataset=Subset(test_dataset, calib_ind)
id_eval_dataset = Subset(test_dataset, id_eval_ind)
rest_test_dataset = Subset(test_dataset, rest_test_ind)


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
test_loader = DataLoader(rest_test_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
calib_loader = DataLoader(calib_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)
id_eval_loader = DataLoader(id_eval_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)


data_iter = iter(train_loader)
images, labels = next(data_iter)
print(f"Dimenzije batch-a slika: {images.shape}")  # Očekujemo [64, 1, 28, 28]
print(f"Dimenzije batch-a labela: {labels.shape}") # Očekujemo [64]
print(f"Calib uzoraka: {len(calib_dataset)} | ID eval uzoraka: {len(id_eval_dataset)} | Ostatak test: {len(rest_test_dataset)}")

Dimenzije batch-a slika: torch.Size([256, 1, 28, 28])
Dimenzije batch-a labela: torch.Size([256])
Calib uzoraka: 2000 | ID eval uzoraka: 2000 | Ostatak test: 112323


In [16]:
import numpy as np
import torch
from torch.utils.data import ConcatDataset, DataLoader, TensorDataset
from torchvision import datasets, transforms
from PIL import Image, ImageDraw


ood_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

kmnist_test = datasets.KMNIST(root='./data', train=False, download=True, transform=ood_transform)
fashion_test = datasets.FashionMNIST(root='./data', train=False, download=True, transform=ood_transform)

n_kmnist = 200
n_fashion = 150

k_idx = torch.randperm(len(kmnist_test), generator=generator)[:n_kmnist].tolist()
f_idx = torch.randperm(len(fashion_test), generator=generator)[:n_fashion].tolist()

kmnist_sub = TensorDataset(
    torch.stack([kmnist_test[i][0] for i in k_idx]),
    torch.full((n_kmnist,), -1, dtype=torch.long)  # -1 označava da ne pripada EMNIST klasama
)
fashion_sub = TensorDataset(
    torch.stack([fashion_test[i][0] for i in f_idx]),
    torch.full((n_fashion,), -1, dtype=torch.long)
)


n_scribbles = 150
scribble_tensors = []

for _ in range(n_scribbles):
    img = Image.new("L", (28, 28), color=0)
    draw = ImageDraw.Draw(img)
    n_strokes = rng.integers(2, 6)
    for _ in range(n_strokes):
        pts = [(int(rng.integers(2, 26)), int(rng.integers(2, 26))) for _ in range(rng.integers(2, 5))]
        draw.line(pts, fill=int(rng.integers(180, 256)), width=int(rng.integers(1, 3)))
    scribble_tensors.append(ood_transform(img))

scribble_sub = TensorDataset(
    torch.stack(scribble_tensors),
    torch.full((n_scribbles,), -1, dtype=torch.long)
)


ood_dataset = ConcatDataset([kmnist_sub, fashion_sub, scribble_sub])
ood_loader = DataLoader(ood_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

print(f"OOD skup spreman: {len(ood_dataset)} uzoraka (KMNIST: {n_kmnist}, Fashion: {n_fashion}, Škrabotine: {n_scribbles})")

OOD skup spreman: 500 uzoraka (KMNIST: 200, Fashion: 150, Škrabotine: 150)


In [17]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channles=1, patch_size=4, emb_dim=64, img_size=28): #64 filtera, in_ch je dimenzija boje
        super().__init__()
        self.patch_size = patch_size

        self.proj = nn.Conv2d( #izmerimo koliko se slazu filteri sa slikom
            in_channels=in_channles,
            out_channels=emb_dim,
            kernel_size=patch_size,
            stride=patch_size
        )
    def forward(self,x):
        x=self.proj(x)
        x=x.flatten(2) # pretvorimo u 'string', cuvamo prve dve dim
        x=x.transpose(1,2) # menjamo mesta 2 i 3 dim, da bi svaki od 49 patcheva dobio vektor od 64 dim// takav nam oblik treba za transformer

        return x


In [18]:
class ViTInputEmbedding(nn.Module):
    def __init__(self, in_channels=1,patch_size=4,emb_dim=64, img_size=28, num_classes=62): #num classes je broj razlicitih slova i brojeva, trebace kasnije
        super().__init__()
        self.patch_embed=PatchEmbedding(in_channels, patch_size, emb_dim, img_size)

        self.cls_token=nn.Parameter(torch.randn(1,1, emb_dim)) #nn.paremeter kaze torchu da se na njemu uci

        num_patches= (img_size//patch_size)**2
        self.pos_embedding= nn.Parameter(torch.randn(1, num_patches + 1, emb_dim))

    def forward(self,x):

        batch_size=x.shape[0]

        x=self.patch_embed(x)

        cls_tokens=self.cls_token.expand(batch_size,-1,-1)

        x=torch.cat((cls_tokens, x), dim=1)

        x=x+ self.pos_embedding # sabiranje pozicionih informacija je backpropagatabilno
        return x




In [19]:
class TransformerBlock(nn.Module):
    def __init__(self, emb_dim=64, num_heads=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1=nn.LayerNorm(emb_dim)

        self.attn= nn.MultiheadAttention(embed_dim=emb_dim, num_heads=num_heads, dropout=dropout, batch_first=True)

        self.norm2 = nn.LayerNorm(emb_dim)

        hidden_dim= int(emb_dim * mlp_ratio)

        self.mlp= nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim,emb_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):

        normed_x= self.norm1(x)

        attn_out, _= self.attn(normed_x, normed_x, normed_x)

        x=x+attn_out

        x=x+ self.mlp(self.norm2(x))

        return x



In [20]:
class VisionTransformer(nn.Module):
    def __init__(self, in_channels=1, patch_size=4, emb_dim=64, img_size=28, num_classes=62, depth=4, num_heads=4, dropout=0.1):
        super().__init__()
        self.embedding=ViTInputEmbedding(in_channels=in_channels, patch_size=patch_size, emb_dim=emb_dim, img_size=img_size, num_classes=num_classes)

        self.blocks= nn.ModuleList([
            TransformerBlock(emb_dim=emb_dim, num_heads=num_heads, dropout=dropout)
            for _ in range(depth)
        ])

        self.norm= nn.LayerNorm(emb_dim)

        self.head= nn.Linear(emb_dim, num_classes)

    def forward(self,x):
        x= self.embedding(x)

        for block in self.blocks:
            x= block(x)

        x=self.norm(x)

        cls_token_final = x[:,0]

        logits= self.head(cls_token_final)

        return logits


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Koristimo urejdaj: {device}")

model = VisionTransformer(
    in_channels=1,
    patch_size=4,
    emb_dim=64,
    img_size=28, 
    num_classes=62,
    depth=4,
    num_heads=4,
    dropout=0.1
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

print(f"Model je kreiran i prebacen na {device}")

Koristimo urejdaj: cuda
Model je kreiran i prebacen na cuda


In [57]:
num_epochs = 10
model.train()

for epoch in range(num_epochs):
    total_loss= 0.0
    correct=0
    total=0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)

        loss= criterion(logits, labels)

        loss.backward()

        #print("Da li je tenzor na GPU-u?", images.is_cuda)
        #print(f"Trenutna VRAM memorija u upotrebi: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        #print(f"Maksimalna VRAM memorija iskorišćena do sada: {torch.cuda.max_memory_allocated() / 1024**2:.2f} MB")
        
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(logits.detach(), 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()


        if batch_idx % 500 == 0:
            print(f"Epoha [{epoch+1}/{num_epochs}] | Batch [{batch_idx}/{len(train_loader)}] | Loss: {loss.item():.4f}")

    epoch_loss= total_loss/ len(train_loader)
    epoch_acc= 100*correct / total
    print(f"==> Kraj Epohe {epoch+1} | Prosečan Loss: {epoch_loss:.4f} | Tačnost (Accuracy): {epoch_acc:.2f}%\n")

Epoha [1/10] | Batch [0/2727] | Loss: 4.3564
Epoha [1/10] | Batch [500/2727] | Loss: 1.3176
Epoha [1/10] | Batch [1000/2727] | Loss: 0.8744
Epoha [1/10] | Batch [1500/2727] | Loss: 0.6578
Epoha [1/10] | Batch [2000/2727] | Loss: 0.6438
Epoha [1/10] | Batch [2500/2727] | Loss: 0.6170
==> Kraj Epohe 1 | Prosečan Loss: 1.0490 | Tačnost (Accuracy): 69.80%

Epoha [2/10] | Batch [0/2727] | Loss: 0.6381
Epoha [2/10] | Batch [500/2727] | Loss: 0.5664
Epoha [2/10] | Batch [1000/2727] | Loss: 0.6137
Epoha [2/10] | Batch [1500/2727] | Loss: 0.4980
Epoha [2/10] | Batch [2000/2727] | Loss: 0.6180
Epoha [2/10] | Batch [2500/2727] | Loss: 0.4674
==> Kraj Epohe 2 | Prosečan Loss: 0.5712 | Tačnost (Accuracy): 80.70%

Epoha [3/10] | Batch [0/2727] | Loss: 0.5516
Epoha [3/10] | Batch [500/2727] | Loss: 0.4490
Epoha [3/10] | Batch [1000/2727] | Loss: 0.4941
Epoha [3/10] | Batch [1500/2727] | Loss: 0.5326
Epoha [3/10] | Batch [2000/2727] | Loss: 0.5282
Epoha [3/10] | Batch [2500/2727] | Loss: 0.5099
==> Kr

In [58]:
torch.save({
    'epoch': num_epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, 'vit_emnist.pth')

In [21]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = VisionTransformer(
    in_channels=1,
    patch_size=4,
    emb_dim=64,
    img_size=28, 
    num_classes=62,
    depth=4,
    num_heads=4,
    dropout=0.1
).to(device)


optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)


checkpoint = torch.load('vit_emnist.pt', map_location=device)


model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

print("Model je uspešno učitan sa diska!")

Model je uspešno učitan sa diska!


C:\Users\Nikola\AppData\Local\Temp\ipykernel_25004\3488416105.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('vit_emnist.pt', map_location=devi

In [8]:
model.eval()

VisionTransformer(
  (embedding): ViTInputEmbedding(
    (patch_embed): PatchEmbedding(
      (proj): Conv2d(1, 64, kernel_size=(4, 4), stride=(4, 4))
    )
  )
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
      )
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=64, out_features=256, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=256, out_features=64, bias=True)
        (4): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=64, out_features=62, bias=True)
)

In [9]:
correct = 0
total = 0

with torch.no_grad():  # bez gradijenata
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total
print(f"Tačnost modela na test skupu: {test_accuracy:.2f}%")

Tačnost modela na test skupu: 86.31%


In [22]:
import tkinter as tk
from PIL import Image, ImageDraw, ImageOps
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import clear_output
from scipy.ndimage import center_of_mass

root = tk.Tk()
root.title("Vit EMINST test")

canvas_size=280
canvas= tk.Canvas(root, width=canvas_size, height=canvas_size, bg="black")
canvas.pack(pady=10)

app_image = Image.new("L", (canvas_size, canvas_size), color=0)
app_draw = ImageDraw.Draw(app_image)


In [24]:

def get_emnist_char(label):
    if label < 10:
        return str(label)
    elif label < 36:
        return chr(ord('A') + (label - 10))
    else:
        return chr(ord('a') + (label - 36))

def paint(event):
    x, y = event.x, event.y
    r = 15
  
    canvas.create_oval(x - r, y - r, x + r, y + r, fill="white", outline="white")
    app_draw.ellipse([x - r, y - r, x + r, y + r], fill=255)

canvas.bind("<B1-Motion>", paint)

label_result = tk.Label(root, text="Nacrtaj nešto i klikni Prepoznaj...", font=("Arial", 16, "bold"))
label_result.pack(pady=5)

def clear_canvas():
   
    canvas.delete("all")
    
    
    global app_image, app_draw
    app_image = Image.new("L", (canvas_size, canvas_size), color=0)
    app_draw = ImageDraw.Draw(app_image)
    
    
    label_result.config(text="Nacrtaj nešto i klikni Prepoznaj...")
    
btn_predict = tk.Button(root, text="Prepoznaj slovo", command=predict_drawing, font=("Arial", 12), bg="lightgreen")
btn_predict.pack(side=tk.LEFT, padx=20, pady=10)
btn_clear = tk.Button(root, text="Obriši platno", command=clear_canvas, font=("Arial", 12))
btn_clear.pack(pady=10)

root.mainloop()

--- MONTE CARLO DROPOUT (20 PROLAZA) ---
1. Klasa indeks 2 ('2'): Srednja verovatnoća = 84.1% (Nesigurnost/Std = ±11.1%)
2. Klasa indeks 35 ('Z'): Srednja verovatnoća = 15.2% (Nesigurnost/Std = ±10.8%)
3. Klasa indeks 61 ('z'): Srednja verovatnoća = 0.2% (Nesigurnost/Std = ±0.1%)
4. Klasa indeks 21 ('L'): Srednja verovatnoća = 0.1% (Nesigurnost/Std = ±0.1%)
5. Klasa indeks 39 ('d'): Srednja verovatnoća = 0.1% (Nesigurnost/Std = ±0.1%)

>>> REZULTAT: Prihvaćeno slovo '2'


In [12]:
def predict_drawing():
    
    #plt.clf()
    #plt.close('all')
    img_resized = app_image.resize((28,28), Image.Resampling.LANCZOS)
    #img_resized = img_resized.point(lambda p: 255 if p > 50 else 0, mode='1').convert('L')
    img_resized= img_resized.rotate(270, expand=True)
    img_resized=ImageOps.mirror(img_resized)

    

    img_np = np.array(img_resized, dtype=np.float32) / 255.0

    if img_np.sum() > 0:
        cy, cx = center_of_mass(img_np)
        rows, cols = img_np.shape
        shift_y = int(rows / 2.0 - cy)
        shift_x = int(cols / 2.0 - cx)
        img_np = np.roll(img_np, shift_y, axis=0)
        img_np = np.roll(img_np, shift_x, axis=1)
    np.set_printoptions(linewidth=np.inf, precision=1, suppress=True)
    #print(img_np)
    # plt.figure(figsize=(3, 3))
    # plt.imshow(img_resized, cmap='gray')
    # plt.title("Ovo vidi model (28x28)")
    # plt.axis('off')
    # plt.show()
    
    #img_np = 1.0 - img_np
    #print(f"Maksimalna vrednost piksela na slici: {img_np.max()} ")
    tensor_img = torch.tensor(img_np, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    tensor_img = torch.rot90(tensor_img, 1, [2, 3])
    tensor_img = torch.flip(tensor_img, [3])
    # Normalizacija na [-1, 1]
    tensor_img = (tensor_img - 0.5) / 0.5
    
    model.eval()

    with torch.no_grad():
        # outputs= model(tensor_img)
        # #print("Raw logits:", outputs[0][:10])
        # probs = F.softmax(outputs, dim=1)
        # #print("Raw probs:", probs[0][:])
        # confidance, predicted = torch.max(probs,1)
        # print("predicted.item:", predicted.item())
        outputs = model(tensor_img)
        probabilities = F.softmax(outputs, dim=1)
        
        # Uzimamo TOP 5 predviđanja umesto samo jednog
        top5_prob, top5_indices = torch.topk(probabilities, 5, dim=1)

    #print("--- TOP 5 PREDIKCIJA ---")
    #for i in range(5):
        #idx = top5_indices[0][i].item()
        #prob = top5_prob[0][i].item() * 100
        #char = get_emnist_char(idx)
        #print(f"{i+1}. Klasa indeks {idx} ('{char}'): {prob:.1f}%")



    pred_idx= top5_indices[0][0].item()
    conf_val= top5_prob[0][0].item()*100

    char_result = get_emnist_char(pred_idx)
    #print(f"Predviđeni indeks: {pred_idx}, Karakter: {char_result}")
    
    label_result.config(text=f"Predikcija: '{char_result}' ({conf_val:.1f}% pouzdanost)")


 

In [ ]:
import torchvision.datasets as datasets
import matplotlib.pyplot as plt
import numpy as np


emnist_train = datasets.EMNIST(
    root='./data', 
    split='byclass', 
    train=True, 
    download=True
)


sample_idx = 1011
img_pil, label_idx = emnist_train[sample_idx]


img_np = np.array(img_pil, dtype=np.float32) / 255.0


np.set_printoptions(precision=1, suppress=True, linewidth=120)
print(f"--- NUMPY MATRICA PRAVOG EMNIST PRIMERA (Indeks klase: {label_idx}) ---")
print(img_np)


plt.figure(figsize=(3, 3))
plt.imshow(img_np, cmap='gray')


char_name = get_emnist_char(label_idx)
plt.title(f"Pravi EMNIST -> Klasa {label_idx} ('{char_name}')")
plt.axis('off')
plt.show()

In [23]:
def predict_drawing():
    
    img_resized = app_image.resize((28,28), Image.Resampling.LANCZOS)
    img_resized= img_resized.rotate(270, expand=True)
    img_resized=ImageOps.mirror(img_resized)

    

    img_np = np.array(img_resized, dtype=np.float32) / 255.0

    if img_np.sum() > 0:
        cy, cx = center_of_mass(img_np)
        rows, cols = img_np.shape
        shift_y = int(rows / 2.0 - cy)
        shift_x = int(cols / 2.0 - cx)
        img_np = np.roll(img_np, shift_y, axis=0)
        img_np = np.roll(img_np, shift_x, axis=1)
    np.set_printoptions(linewidth=np.inf, precision=1, suppress=True)
    
    
    tensor_img = torch.tensor(img_np, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
    tensor_img = torch.rot90(tensor_img, 1, [2, 3])
    tensor_img = torch.flip(tensor_img, [3])
    tensor_img = (tensor_img - 0.5) / 0.5
    
    # ovde divergira
    
    model.train()
    n_passes= 20
    all_probs=[]


    with torch.no_grad():
        for _ in range(n_passes):

            outputs = model(tensor_img)
            #print(f"outputs: {outputs.shape}")
            #print(outputs)
            probs = F.softmax(outputs, dim=1)
            #print(f"probs: {probs.shape}")
            all_probs.append(probs.cpu().numpy())
            #print(f"allprobs: {len(all_probs)}")
            #print(all_probs)


    all_probs =np.vstack(all_probs)
    #print(f"allprobs: {len(all_probs)}")
    #print(all_probs)

    mean_probs = np.mean(all_probs, 0)
    #print(f"mean_probs: {len(mean_probs)}")
    uncertainty= np.std(all_probs, 0)

    top5_ind = np.argsort(mean_probs)[::-1][:5]
    print(f"--- MONTE CARLO DROPOUT ({n_passes} PROLAZA) ---")
    for i, idx in enumerate(top5_ind):
        prob = mean_probs[idx] * 100
        unc = uncertainty[idx] * 100
        char = get_emnist_char(idx)
        print(f"{i+1}. Klasa indeks {idx} ('{char}'): Srednja verovatnoća = {prob:.1f}% (Nesigurnost/Std = ±{unc:.1f}%)")
    
    best_idx = top5_ind[0]
    best_conf = mean_probs[best_idx] * 100
    best_unc = uncertainty[best_idx] * 100
    best_char = get_emnist_char(best_idx)


    CONFIDANCE_THRESHOLD = 50.0
    UNCERTAINTY_THRESHOLD= 15.0

    if best_conf<CONFIDANCE_THRESHOLD or best_unc>UNCERTAINTY_THRESHOLD:
        label_result.config(text=f"Nepoznato / Nesigurno (Conf: {best_conf:.1f}%, Std: ±{best_unc:.1f}%)")
        print("\n>>> REZULTAT: Model odbacio unos jer nije dovoljno siguran!")
    else:
        label_result.config(text=f"Predikcija: '{best_char}' ({best_conf:.1f}%, ±{best_unc:.1f}%)")
        print(f"\n>>> REZULTAT: Prihvaćeno slovo '{best_char}'")


 

In [25]:
def run_vanilla_vit(model, loader, device):
    
    model.eval()
    all_preds = []
    all_labels = []
    all_rejected = []
    all_ood_proxy = []
    all_unc = []


    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            probs = F.softmax(logits, dim=1)

            max_probs, preds = torch.max(probs, dim=1)
            rejected = torch.zeros_like(preds, dtype=torch.bool)  # Ništa ne odbija
            ood_proxy = 1.0 - max_probs

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())
            all_rejected.append(rejected.cpu().numpy())
            all_ood_proxy.append(ood_proxy.cpu().numpy())
            all_unc.append(ood_proxy.cpu().numpy())

    return (
        np.concatenate(all_preds),
        np.concatenate(all_labels),
        np.concatenate(all_rejected),
        np.concatenate(all_ood_proxy),
        np.concatenate(all_unc),
    )

In [26]:
def enable_mc_dropout(model): #dropout samo na dropout slojeve
   
    model.eval()
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.train()


def run_mc_dropout_vit(model, loader, device, n_passes: int = 20, unc_threshold: float = 0.004):
   
    enable_mc_dropout(model)

    all_preds = []
    all_labels = []
    all_rejected = []
    all_ood_proxy = []
    all_unc=[]

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

           
            mc_probs_list = []
            for _ in range(n_passes):
                logits = model(images)
                probs = F.softmax(logits, dim=1)
                mc_probs_list.append(probs)

            mc_probs = torch.stack(mc_probs_list, dim=0)  # Oblik: [n_passes, batch_size, num_classes]
            mean_probs = mc_probs.mean(dim=0)
            _, preds = torch.max(mean_probs, dim=1)

            
            std_per_class = mc_probs.std(dim=0)
            uncertainty = std_per_class.mean(dim=1)

            rejected = uncertainty > unc_threshold
            ood_proxy = uncertainty

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())
            all_rejected.append(rejected.cpu().numpy())
            all_ood_proxy.append(ood_proxy.cpu().numpy())
            all_unc.append(uncertainty.cpu().numpy())
            

   
    model.eval()

    return (
        np.concatenate(all_preds),
        np.concatenate(all_labels),
        np.concatenate(all_rejected),
        np.concatenate(all_ood_proxy),
        np.concatenate(all_unc),
    )

In [27]:
from sklearn.metrics import roc_auc_score

def evaluate_machine(
    name: str,
    id_preds: np.ndarray,
    id_labels: np.ndarray,
    id_rejected: np.ndarray,
    id_uncertainty: np.ndarray,
    ood_rejected: np.ndarray,
    ood_uncertainty: np.ndarray,
    id_ood_scores: np.ndarray,
    ood_ood_scores: np.ndarray,
):
    
    id_raw_acc = float(np.mean(id_preds == id_labels))

    accepted_mask = ~id_rejected
    id_coverage = float(np.mean(accepted_mask))
    if np.sum(accepted_mask) > 0:
        id_selective_acc = float(np.mean(id_preds[accepted_mask] == id_labels[accepted_mask]))
    else:
        id_selective_acc = 0.0

    ood_rejection_rate = float(np.mean(ood_rejected))

    y_true_ood = np.concatenate([np.zeros(len(id_ood_scores)), np.ones(len(ood_ood_scores))])
    y_scores_ood = np.concatenate([id_ood_scores, ood_ood_scores])
    auroc = float(roc_auc_score(y_true_ood, y_scores_ood))

    results = {
        "machine": name,
        "id_raw_acc": id_raw_acc,
        "id_coverage": id_coverage,
        "id_selective_acc": id_selective_acc,
        "id_unc_mean": float(np.mean(id_uncertainty)),
        "ood_unc_mean": float(np.mean(ood_uncertainty)),
        "ood_rejection_rate": ood_rejection_rate,
        "id_vs_ood_auroc": auroc,
    }

    print(f"\n=== {name} ===")
    print(f"ID Sirova tačnost:      {id_raw_acc * 100:6.2f}%")
    print(f"ID Pokrivenost (Cov):   {id_coverage * 100:6.2f}%")
    print(f"ID Selektivna tačnost:  {id_selective_acc * 100:6.2f}%")
    print(f"Nesigurnost (ID / OOD): {np.mean(id_uncertainty):.5f} / {np.mean(ood_uncertainty):.5f}")
    print(f"OOD Odbacivanje:        {ood_rejection_rate * 100:6.2f}%")
    print(f"ID vs OOD AUROC:        {auroc:6.4f}")

    return results

In [37]:
import pandas as pd

all_machine_results = []

# ==========================================
# 1. MAŠINA 1: Vanilla ViT (bez odbijanja)
# ==========================================
print("\n[Mašina 1 - Vanilla ViT] Pokrećem osnovni prolaz bez filtera odbijanja...")
m1_id_preds, m1_id_labels, m1_id_rej, m1_id_unc, m1_id_proxy = run_vanilla_vit(
    model, id_eval_loader, device
)
_, _, m1_ood_rej, m1_ood_unc, m1_ood_proxy = run_vanilla_vit(
    model, ood_loader, device
)

res_m1 = evaluate_machine(
    name="Mašina 1: Vanilla ViT (bez odbijanja)",
    id_preds=m1_id_preds,
    id_labels=m1_id_labels,
    id_rejected=m1_id_rej,
    id_uncertainty=m1_id_unc,
    ood_rejected=m1_ood_rej,
    ood_uncertainty=m1_ood_unc,
    id_ood_scores=m1_id_proxy,
    ood_ood_scores=m1_ood_proxy,
)
all_machine_results.append(res_m1)

# ==========================================
# 2. MAŠINA 2: ViT + MC Dropout
# ==========================================
N_PASSES = 20
UNC_THRESHOLD = 0.004  # realniji prag (0.05 je ranije bio previsoko za mean std)

print(
    f"\n[Mašina 2 - MC Dropout] Pokrećem {N_PASSES} stohastičkih prolaza po uzorku "
    f"(prag nesigurnosti = {UNC_THRESHOLD})..."
)
m2_id_preds, m2_id_labels, m2_id_rej, m2_id_unc, m2_id_proxy = run_mc_dropout_vit(
    model,
    id_eval_loader,
    device,
    n_passes=N_PASSES,
    unc_threshold=UNC_THRESHOLD,
)
_, _, m2_ood_rej, m2_ood_unc, m2_ood_proxy = run_mc_dropout_vit(
    model,
    ood_loader,
    device,
    n_passes=N_PASSES,
    unc_threshold=UNC_THRESHOLD,
)

print(
    "[Dijagnostika nesigurnosti] "
    f"Poznati (ID) prosek / max: {m2_id_unc.mean():.5f} / {m2_id_unc.max():.5f} | "
    f"Nepoznati (OOD) prosek / max: {m2_ood_unc.mean():.5f} / {m2_ood_unc.max():.5f}"
)
print(
    f"[Uticaj praga {UNC_THRESHOLD}] "
    f"Odbijeno poznatih (ID): {m2_id_rej.mean() * 100:.2f}% | "
    f"Odbijeno nepoznatih (OOD): {m2_ood_rej.mean() * 100:.2f}%"
)

res_m2 = evaluate_machine(
    name=f"Mašina 2: MC Dropout (N={N_PASSES}, prag={UNC_THRESHOLD})",
    id_preds=m2_id_preds,
    id_labels=m2_id_labels,
    id_rejected=m2_id_rej,
    id_uncertainty=m2_id_unc,
    ood_rejected=m2_ood_rej,
    ood_uncertainty=m2_ood_unc,
    id_ood_scores=m2_id_proxy,
    ood_ood_scores=m2_ood_proxy,
)
all_machine_results.append(res_m2)

print(
    f"\n[Mašina 3 - MC Dropout + Temp Scaling] Pokrećem sa T={optimal_T:.4f}, "
    f"N={N_PASSES}, prag={UNC_THRESHOLD}..."
)
m3_id_preds, m3_id_labels, m3_id_rej, m3_id_unc, m3_id_proxy = run_mc_temp_vit(
    model,
    id_eval_loader,
    device,
    temperature=optimal_T,
    n_passes=N_PASSES,
    unc_threshold=UNC_THRESHOLD,
)
_, _, m3_ood_rej, m3_ood_unc, m3_ood_proxy = run_mc_temp_vit(
    model,
    ood_loader,
    device,
    temperature=optimal_T,
    n_passes=N_PASSES,
    unc_threshold=UNC_THRESHOLD,
)

print(
    "[Poređenje nesigurnosti M2 -> M3] "
    f"ID prosek: {m2_id_unc.mean():.5f} -> {m3_id_unc.mean():.5f} | "
    f"OOD prosek: {m2_ood_unc.mean():.5f} -> {m3_ood_unc.mean():.5f}"
)
print(
    f"[Odbijanje uz prag {UNC_THRESHOLD} i T={optimal_T:.4f}] "
    f"Odbijeno ID: {m3_id_rej.mean() * 100:.2f}% | "
    f"Odbijeno OOD: {m3_ood_rej.mean() * 100:.2f}%"
)

res_m3 = evaluate_machine(
    name=f"Mašina 3: MC + Temp (T={optimal_T:.2f}, prag={UNC_THRESHOLD})",
    id_preds=m3_id_preds,
    id_labels=m3_id_labels,
    id_rejected=m3_id_rej,
    id_uncertainty=m3_id_unc,
    ood_rejected=m3_ood_rej,
    ood_uncertainty=m3_ood_unc,
    id_ood_scores=m3_id_proxy,
    ood_ood_scores=m3_ood_proxy,
)
all_machine_results.append(res_m3)

res_m4 = evaluate_machine(
    name=f"Mašina 4: MC + Temp + Energy OOD (prag={ood_threshold:.2f})",
    id_preds=m4_id_preds,
    id_labels=m4_id_labels,
    id_rejected=m4_id_rej,
    id_uncertainty=m4_id_unc,
    ood_rejected=m4_ood_rej,
    ood_uncertainty=m4_ood_unc,
    id_ood_scores=m4_id_ood,
    ood_ood_scores=m4_ood_score,
)
all_machine_results.append(res_m4)

df_results = pd.DataFrame(all_machine_results).set_index("machine")
df_pretty = pd.DataFrame({
    "ID Sirova tačnost": (df_results["id_raw_acc"] * 100).round(2).astype(str) + "%",
    "ID Pokrivenost": (df_results["id_coverage"] * 100).round(2).astype(str) + "%",
    "ID Selektivna tačnost": (df_results["id_selective_acc"] * 100).round(2).astype(str) + "%",
    "Prosek unc (ID)": df_results["id_unc_mean"].round(5),
    "Prosek unc (OOD)": df_results["ood_unc_mean"].round(5),
    "Odbijeno OOD": (df_results["ood_rejection_rate"] * 100).round(2).astype(str) + "%",
    "AUROC": df_results["id_vs_ood_auroc"].round(4),
})

print(df_pretty.to_string())


[Mašina 1 - Vanilla ViT] Pokrećem osnovni prolaz bez filtera odbijanja...

=== Mašina 1: Vanilla ViT (bez odbijanja) ===
ID Sirova tačnost:       86.60%
ID Pokrivenost (Cov):   100.00%
ID Selektivna tačnost:   86.60%
Nesigurnost (ID / OOD): 0.13064 / 0.46689
OOD Odbacivanje:          0.00%
ID vs OOD AUROC:        0.8908

[Mašina 2 - MC Dropout] Pokrećem 20 stohastičkih prolaza po uzorku (prag nesigurnosti = 0.004)...
[Dijagnostika nesigurnosti] Poznati (ID) prosek / max: 0.00114 / 0.01313 | Nepoznati (OOD) prosek / max: 0.00858 / 0.01659
[Uticaj praga 0.004] Odbijeno poznatih (ID): 6.60% | Odbijeno nepoznatih (OOD): 91.40%

=== Mašina 2: MC Dropout (N=20, prag=0.004) ===
ID Sirova tačnost:       86.40%
ID Pokrivenost (Cov):    93.40%
ID Selektivna tačnost:   88.60%
Nesigurnost (ID / OOD): 0.00114 / 0.00858
OOD Odbacivanje:         91.40%
ID vs OOD AUROC:        0.9741

[Mašina 3 - MC Dropout + Temp Scaling] Pokrećem sa T=1.0262, N=20, prag=0.004...
[Poređenje nesigurnosti M2 -> M3] ID

In [29]:
def fit_temperature(model, calib_loader, device, max_iter: int = 50, lr:float=0.01):
    model.eval()

    logits_list=[]
    labels_list=[]
    with torch.no_grad():
        for images, labels in calib_loader:
            images= images.to(device)
            logits_list.append(model(images))
            labels_list.append(labels.to(device))

    calib_logits= torch.cat(logits_list, dim=0)
    calib_labels= torch.cat(labels_list, dim= 0)

    log_temp = nn.Parameter(torch.zeros(1, device=device))
    optimizer = torch.optim.LBFGS([log_temp], lr=lr, max_iter=max_iter)

    def closure():
        optimizer.zero_grad()
        temp = torch.exp(log_temp)
        loss = F.cross_entropy(calib_logits / temp, calib_labels)
        loss.backward()
        return loss
    
    optimizer.step(closure)

    optimal_t = float(torch.exp(log_temp).detach().item())
    with torch.no_grad():
        nll_before = float(F.cross_entropy(calib_logits, calib_labels).item())
        nll_after = float(F.cross_entropy(calib_logits / optimal_t, calib_labels).item())

    print(f"Optimalna temperatura T: {optimal_t:.4f}")
    print(f"Calib NLL pre / posle T: {nll_before:.4f} -> {nll_after:.4f}")
    return optimal_t


optimal_T = fit_temperature(model, calib_loader, device)


Optimalna temperatura T: 1.0262
Calib NLL pre / posle T: 0.3607 -> 0.3601


In [ ]:
def run_mc_temp_vit(
    model,
    loader,
    device,
    temperature: float,
    n_passes: int = 20,
    unc_threshold: float = 0.004,
):
    enable_mc_dropout(model)
    all_preds=[]
    all_labels =[] 
    all_rejected = []
    all_unc=[]
    all_ood_proxy = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            mc_probs = torch.stack([F.softmax(model(images) / temperature, dim=1) for _ in range(n_passes)], dim=0)
    mean_probs = mc_probs.mean(dim=0)
    _, preds = torch.max(mean_probs, dim=1)

    uncertainty = mc_probs.std(dim=0).mean(dim=1)
    rejected = uncertainty > unc_threshold
    ood_proxy = uncertainty

    all_preds.append(preds.cpu().numpy())
    all_labels.append(labels.numpy())
    all_rejected.append(rejected.cpu().numpy())
    all_unc.append(uncertainty.cpu().numpy())
    all_ood_proxy.append(ood_proxy.cpu().numpy())

    model.eval()

    return (
        np.concatenate(all_preds),
        np.concatenate(all_labels),
        np.concatenate(all_rejected),
        np.concatenate(all_unc),
        np.concatenate(all_ood_proxy),
    )

In [35]:
def compute_energy_ood_score(logits: torch.Tensor, temp: float = 1.0) -> torch.Tensor:
    return -temp * torch.logsumexp(logits / temp, dim=1)

def calibrate_ood_threshold(model, calib_loader, device, percentile: float=95, temp: float=1.0)-> float:
    model.eval()
    scores=[]
    with torch.no_grad():
        for images, _ in calib_loader:
            logits=model(images.to(device))
            scores.append(compute_energy_ood_score(logits, temp=temp).cpu().numpy())
    return float(np.percentile(np.concatenate(scores), percentile))
ood_threshold = calibrate_ood_threshold(model, calib_loader, device, percentile=95.0, temp=optimal_T)
print(f"OOD prag: {ood_threshold:.4f}")

OOD prag: -10.6294


In [33]:
def run_mc_temp_ood_vit(
    model,
    loader,
    device,
    temperature: float,
    ood_threshold: float,
    n_passes: int = 20,
    unc_threshold: float = 0.004,
):
    enable_mc_dropout(model)
    all_preds, all_labels, all_rejected = [], [], []
    all_unc, all_ood_scores = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            mc_logits = torch.stack([model(images) / temperature for _ in range(n_passes)], dim=0)
            mc_probs = F.softmax(mc_logits, dim=-1)
            mean_probs = mc_probs.mean(dim=0)
            _, preds = torch.max(mean_probs, dim=1)

            uncertainty = mc_probs.std(dim=0).mean(dim=1)
            ood_score = compute_energy_ood_score(mc_logits.mean(dim=0), temp=temperature)
            rejected = (uncertainty > unc_threshold) | (ood_score > ood_threshold)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.numpy())
            all_rejected.append(rejected.cpu().numpy())
            all_unc.append(uncertainty.cpu().numpy())
            all_ood_scores.append(ood_score.cpu().numpy())

    model.eval()

    return (
        np.concatenate(all_preds),
        np.concatenate(all_labels),
        np.concatenate(all_rejected),
        np.concatenate(all_unc),
        np.concatenate(all_ood_scores),
    )

In [36]:
m4_id_preds, m4_id_labels, m4_id_rej, m4_id_unc, m4_id_ood = run_mc_temp_ood_vit(
    model,
    id_eval_loader,
    device,
    temperature=optimal_T,
    ood_threshold=ood_threshold,
    n_passes=N_PASSES,
    unc_threshold=UNC_THRESHOLD,
)
_, _, m4_ood_rej, m4_ood_unc, m4_ood_score = run_mc_temp_ood_vit(
    model,
    ood_loader,
    device,
    temperature=optimal_T,
    ood_threshold=ood_threshold,
    n_passes=N_PASSES,
    unc_threshold=UNC_THRESHOLD,
)

print(f"ID skor  -> prosek: {m4_id_ood.mean():.4f}, medijana: {np.median(m4_id_ood):.4f}")
print(f"OOD skor -> prosek: {m4_ood_score.mean():.4f}, medijana: {np.median(m4_ood_score):.4f}")

ID skor  -> prosek: -12.9275, medijana: -12.9157
OOD skor -> prosek: -7.9858, medijana: -7.8694
